# 13 — Compare: RULER (kvpress vs vLLM)

Side-by-side comparison of KV cache compression on the
[RULER benchmark](https://huggingface.co/datasets/simonjegou/ruler)
from notebooks 07 (kvpress) and 08 (vLLM).

Both frameworks test **KeyDiffPress**-based compression with two
decoding strategies:
- **full_replacement** — full KV cache replacement after prefill
- **filtering** — token filtering during decoding

Visualizations:
1. Per-task heatmaps (kvpress vs vLLM, one figure per context length)
2. Average score vs compression ratio (line chart)
3. Summary table

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

ALGORITHMS = ['full_replacement', 'filtering']
SHORT_NAMES = {
    'full_replacement': 'Full Replacement',
    'filtering': 'Filtering',
    'no_press': 'No compression',
}

## 1. Load Results

In [ ]:
def load_ruler_metrics(path, framework):
    with open(path) as f:
        raw = json.load(f)
    rows = []
    for key, tasks in raw.items():
        parts = key.split('__')
        press = parts[0]
        ratio = float(parts[1])
        ctx = int(float(parts[2]))
        for task, scores in tasks.items():
            rows.append({
                'framework': framework,
                'press': press,
                'compression_ratio': ratio,
                'context_length': ctx,
                'task': task,
                'string_match': scores['string_match'],
            })
    return pd.DataFrame(rows)

kvpress_df = load_ruler_metrics('results/kvpress_ruler/metrics.json', 'kvpress')
vllm_df = load_ruler_metrics('results/vllm_ruler/metrics.json', 'vllm')
df = pd.concat([kvpress_df, vllm_df], ignore_index=True)

print(f'Loaded {len(kvpress_df)} kvpress + {len(vllm_df)} vllm = {len(df)} total')
print(f'Frameworks: {sorted(df["framework"].unique())}')
print(f'Algorithms: {sorted(df["press"].unique())}')
print(f'Context lengths: {sorted(df["context_length"].unique())}')
print(f'Tasks ({df["task"].nunique()}): {sorted(df["task"].unique())}')

## 2. Per-task Heatmaps

One figure per context length, with kvpress and vLLM side by side
for each compression algorithm. Same colormap and annotation style
as notebook 06.

In [ ]:
for ctx in sorted(df['context_length'].unique()):
    ctx_df = df[(df['context_length'] == ctx) & (df['press'] != 'no_press')]
    tasks = sorted(ctx_df['task'].unique())

    fig, axes = plt.subplots(
        len(ALGORITHMS), 2,
        figsize=(14, 4 * len(ALGORITHMS)),
        squeeze=False,
    )

    for row_idx, algo in enumerate(ALGORITHMS):
        for col_idx, fw in enumerate(['kvpress', 'vllm']):
            sub = ctx_df[(ctx_df['press'] == algo) & (ctx_df['framework'] == fw)]
            ratios = sorted(sub['compression_ratio'].unique())

            heatmap_data = np.zeros((len(ratios), len(tasks)))
            for i, r in enumerate(ratios):
                for j, t in enumerate(tasks):
                    cell = sub[(sub['compression_ratio'] == r) & (sub['task'] == t)]
                    heatmap_data[i, j] = cell['string_match'].values[0] if not cell.empty else 0

            ax = axes[row_idx][col_idx]
            ax.imshow(heatmap_data, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
            ax.set_xticks(range(len(tasks)))
            ax.set_xticklabels(tasks, rotation=45, ha='right', fontsize=8)
            ax.set_yticks(range(len(ratios)))
            ax.set_yticklabels([f'{r:.2f}' for r in ratios])
            ax.set_ylabel('Compression Ratio')
            ax.set_title(f'{fw} — {SHORT_NAMES[algo]}')

            for i in range(len(ratios)):
                for j in range(len(tasks)):
                    val = heatmap_data[i, j]
                    ax.text(
                        j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=8, color='white' if val < 50 else 'black',
                    )

    fig.suptitle(
        f'RULER ({ctx}): kvpress vs vLLM — Per-task String Match',
        fontsize=14, y=1.02,
    )
    fig.tight_layout()
    fig.savefig(f'results/compare_ruler_{ctx}_heatmaps.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Average Score vs Compression Ratio

Averaged across all context lengths and tasks.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'kvpress': 'tab:blue', 'vllm': 'tab:orange'}
styles = {'full_replacement': '-', 'filtering': '--'}
markers = {'full_replacement': 'o', 'filtering': 's'}

for fw in ['kvpress', 'vllm']:
    for algo in ALGORITHMS:
        sub = df[(df['framework'] == fw) & (df['press'] == algo)]
        avg = sub.groupby('compression_ratio')['string_match'].mean().sort_index()
        ax.plot(
            avg.index, avg.values,
            marker=markers[algo], linewidth=2,
            color=colors[fw], linestyle=styles[algo],
            label=f'{fw} — {SHORT_NAMES[algo]}',
        )

for fw in ['kvpress', 'vllm']:
    baseline = df[(df['framework'] == fw) & (df['press'] == 'no_press')]
    if not baseline.empty:
        score = baseline['string_match'].mean()
        ax.axhline(
            score, color=colors[fw], linestyle=':', linewidth=1.5, alpha=0.5,
            label=f'{fw} — No compression ({score:.1f}%)',
        )

ax.set_xlabel('Compression Ratio')
ax.set_ylabel('Average String Match (%)')
ax.set_title('RULER: Average Score vs Compression Ratio')
ax.legend()
fig.tight_layout()
fig.savefig('results/compare_ruler_score_vs_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Summary

In [ ]:
summary = (
    df.groupby(['framework', 'press', 'compression_ratio'])
    .agg(avg_score=('string_match', 'mean'))
    .round(2)
)
print(summary.to_string())